# CineOS — RealESRGAN Super Resolution Worker

Upscales images using RealESRGAN on a free Colab GPU.
Exposes a REST API for the CineOS Cloud Worker Bridge.

**Runtime: GPU (T4)**

In [ ]:
#@title 1. Configuration
NGROK_AUTH_TOKEN = "" #@param {type:"string"}
CINEOS_API_KEY = "" #@param {type:"string"}
INACTIVITY_TIMEOUT_MINUTES = 15 #@param {type:"integer"}
API_PORT = 8299
assert NGROK_AUTH_TOKEN, "Set NGROK_AUTH_TOKEN"
print(f"Config: port={API_PORT}")

In [ ]:
#@title 2. Install RealESRGAN
import subprocess, sys, os
def run(cmd):
    subprocess.run(cmd, shell=True, capture_output=True)

run("apt-get update -qq && apt-get install -y -qq libgl1-mesa-glx libglib2.0-0")
if not os.path.exists("/content/Real-ESRGAN"):
    run("git clone --depth 1 https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN")
    run(f"{sys.executable} -m pip install -q basicsr facexlib gfpgan")
    run(f"{sys.executable} -m pip install -q -e /content/Real-ESRGAN")
print("RealESRGAN installed")

In [ ]:
#@title 3. Start REST API + ngrok
import threading, uuid, hashlib, base64, time, signal, os, subprocess, sys
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

api = Flask(__name__); CORS(api)
_last_activity = time.monotonic(); _shutting_down = False; _jobs = {}; _count = 0; _t0 = time.monotonic()

def _inactivity():
    global _shutting_down
    while not _shutting_down:
        if time.monotonic() - _last_activity > INACTIVITY_TIMEOUT_MINUTES * 60:
            _shutting_down = True; os.kill(os.getpid(), signal.SIGTERM); break
        time.sleep(30)
threading.Thread(target=_inactivity, daemon=True).start()

@api.route("/health")
def health(): return jsonify({"status":"healthy","uptime":round(time.monotonic()-_t0,1)})

@api.route("/warmup", methods=["POST"])
def warmup():
    global _last_activity; _last_activity = time.monotonic()
    return jsonify({"status":"ok"})

@api.route("/job", methods=["POST"])
def job():
    global _last_activity, _count; _last_activity = time.monotonic(); _count += 1
    if CINEOS_API_KEY and request.headers.get("X-Api-Key") != CINEOS_API_KEY:
        return jsonify({"error":"Unauthorized"}), 401
    data = request.get_json(); tid = data.get("task_id", str(uuid.uuid4()))
    payload = data.get("payload", {}); img_path = payload.get("image_path","")
    scale = payload.get("scale", 4); model = payload.get("model","realesrgan-x4plus")
    _jobs[tid] = {"status":"processing"}
    threading.Thread(target=_upscale, args=(tid, img_path, scale, model), daemon=True).start()
    return jsonify({"task_id": tid, "status": "processing"})

@api.route("/status/<tid>")
def status(tid): return jsonify(_jobs.get(tid, {"error":"not found"}))

def _upscale(tid, img_path, scale, model):
    try:
        out = f"/content/output/{tid}_sr.png"; os.makedirs("/content/output", exist_ok=True)
        subprocess.run([sys.executable, "-m", "realesrgan.inference_realesrgan",
            "-i", img_path, "-o", out, "-s", str(scale), "-n", model],
            capture_output=True, text=True, timeout=120, check=True)
        with open(out, "rb") as f: ib = f.read()
        _jobs[tid] = {"status":"completed","result":{"image_base64":base64.b64encode(ib).decode(),"checksum":hashlib.sha256(ib).hexdigest(),"source":"realesrgan_colab"}}
    except Exception as e: _jobs[tid] = {"status":"failed","error":str(e)}

threading.Thread(target=lambda: api.run(host="0.0.0.0", port=API_PORT, debug=False), daemon=True).start()
if NGROK_AUTH_TOKEN: ngrok.set_auth_token(NGROK_AUTH_TOKEN)
url = ngrok.connect(API_PORT, "http").public_url
print(f"\nRealESRGAN worker LIVE: {url}")
print(f"Health: {url}/health")
print(f"Add to .env:  COLAB_ESRGAN_ENDPOINT={url}")

signal.signal(signal.SIGTERM, lambda s,f: (ngrok.kill(), os._exit(0)))
while not _shutting_down: time.sleep(10)